# 01 - Data Collection

## Oman Economic Analysis Project

This notebook collects economic data from the **World Bank API** for Oman and other GCC countries.

### Data Sources
- **World Bank Open Data**: Free access to global development data
- **API Documentation**: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392

### Countries Covered
- Oman (OMN)
- Saudi Arabia (SAU)
- United Arab Emirates (ARE)
- Qatar (QAT)
- Kuwait (KWT)
- Bahrain (BHR)

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.data_collector import OmanDataCollector

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Initialize Data Collector

In [ ]:
# Initialize the collector
collector = OmanDataCollector(cache_dir='../data/raw')

# List available indicators
print("Available Economic Indicators:")
print("=" * 60)
indicators_df = collector.list_available_indicators()
for _, row in indicators_df.iterrows():
    print(f"  {row['code']}: {row['description']}")

## 2. Fetch GCC Economic Data

We'll fetch data for all 6 GCC countries from 2000 to 2023.

In [ ]:
# Fetch all indicators for GCC countries
print("Fetching GCC economic data...")
print("This may take a few minutes...\n")

df = collector.fetch_all_indicators(
    start_year=2000,
    end_year=2023,
    save=True
)

In [ ]:
# Data overview
print("\nData Collection Summary")
print("=" * 60)
print(f"Total records: {len(df):,}")
print(f"Countries: {df['country_name'].nunique()}")
print(f"Indicators: {df['indicator_name'].nunique()}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"\nCountries included:")
for country in df['country_name'].unique():
    count = len(df[df['country_name'] == country])
    print(f"  - {country}: {count:,} records")

In [ ]:
# Preview the data
print("\nSample Data:")
df.head(10)

## 3. Data Quality Check

In [ ]:
# Check for missing values
print("Missing Values Analysis")
print("=" * 60)
print(df.isnull().sum())

# Missing values by indicator
print("\nMissing values by indicator:")
missing_by_indicator = df.groupby('indicator_name')['value'].apply(lambda x: x.isnull().sum())
missing_by_indicator = missing_by_indicator[missing_by_indicator > 0].sort_values(ascending=False)
print(missing_by_indicator)

In [ ]:
# Data coverage by country and indicator
print("\nData Coverage Matrix (% of years with data):")
coverage = df.groupby(['country_name', 'indicator_name'])['value'].apply(
    lambda x: (x.notna().sum() / len(x)) * 100
).unstack()
coverage.round(0)

## 4. Quick Preview: Oman's Key Metrics

In [ ]:
# Filter Oman data
oman_df = df[df['country_code'] == 'OMN'].copy()

# Get latest values
print("Oman - Latest Available Data")
print("=" * 60)

latest = oman_df.dropna(subset=['value']).sort_values('year', ascending=False)
latest = latest.groupby('indicator_name').first().reset_index()

for _, row in latest.iterrows():
    print(f"{row['indicator_name']}:")
    print(f"  Value: {row['value']:,.2f} (Year: {row['year']})\n")

## 5. Save Processed Data

In [ ]:
# Save Oman-specific data
oman_df.to_csv('../data/raw/oman_economic_data.csv', index=False)
print("Saved: ../data/raw/oman_economic_data.csv")

print("\n" + "=" * 60)
print("Data collection complete!")
print("Proceed to the analysis notebooks.")
print("=" * 60)

---

## Next Steps

1. **02_gdp_analysis.ipynb** - GDP and economic growth analysis
2. **03_trade_analysis.ipynb** - Trade and exports analysis
3. **04_gcc_comparison.ipynb** - GCC regional comparison